# Step 2 — Preprocessing

Feature engineering, encoding, scaling, and train/val/test splitting of the synthetic EuroCrop dataset produced in Step 1. Logic lives in `src/preprocessor.py`; this notebook runs it end-to-end and inspects the result.

In [1]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
from data_loader import load_raw_data
from preprocessor import (
    engineer_features,
    encode_categoricals,
    split_features_targets,
    preprocess,
    save_processed,
    TARGETS,
)

pd.set_option("display.max_columns", 40)

## Load raw data

In [2]:
df = load_raw_data()
df.shape

(53305, 29)

## Feature engineering

Derive `Days_Since_Harvest`, `Event_Month`, `Event_Hour`, `Event_Dayofweek` from the event timestamp and harvest date, then drop the raw date columns.

In [3]:
df_fe = engineer_features(df.copy())
df_fe[["Days_Since_Harvest", "Event_Month", "Event_Hour", "Event_Dayofweek"]].describe()

,Days_Since_Harvest,Event_Month,Event_Hour,Event_Dayofweek
count,53305.000000,53305.000000,53305.000000,53305.000000
mean,1432.203996,6.504062,11.516518,2.994653
std,639.312154,3.455424,6.931059,2.005309
min,152.000000,1.000000,0.000000,0.000000
25%,887.000000,3.000000,6.000000,1.000000
50%,1432.000000,7.000000,12.000000,3.000000
75%,1974.000000,10.000000,18.000000,5.000000
max,2697.000000,12.000000,23.000000,6.000000


## Encoding

One-hot encode `Crop_Type`. `Vehicle_Type` is left untouched — it is Target 4 (classification), not a feature.

In [4]:
df_enc = encode_categoricals(df_fe)
[c for c in df_enc.columns if c.startswith("Crop_Type")]

['Crop_Type_Corn', 'Crop_Type_Rice', 'Crop_Type_Wheat']

## Feature / target split

The 4 targets: 3 regression (`Spoilage_Risk`, `Efficiency_Ratio`, `Quality_Maintenance_Ratio`) + 1 classification (`Vehicle_Type`). All are dropped from `X`.

In [5]:
X, y = split_features_targets(df_enc)
X.shape, y.shape

((53305, 29), (53305, 4))

In [6]:
y.describe(include="all")

,Spoilage_Risk,Efficiency_Ratio,Quality_Maintenance_Ratio,Vehicle_Type
count,53305.000000,53305.000000,53305.000000,53305
unique,NaN,NaN,NaN,3
top,NaN,NaN,NaN,Truck
freq,NaN,NaN,NaN,22265
mean,25.039335,82.467040,77.505832,NaN
std,13.449138,12.964453,11.298844,NaN
min,0.000000,0.000000,2.220000,NaN
25%,15.240000,75.480000,70.970000,NaN
50%,23.230000,84.630000,78.690000,NaN
75%,33.240000,91.950000,85.440000,NaN


## Full pipeline: split + scale

70/15/15 train/val/test split, `StandardScaler` fit on train only and applied to all splits.

In [7]:
splits = preprocess(df)
{k: v.shape for k, v in splits.items() if hasattr(v, "shape")}

{'X_train': (37313, 29),
 'X_val': (7996, 29),
 'X_test': (7996, 29),
 'y_train': (37313, 4),
 'y_val': (7996, 4),
 'y_test': (7996, 4)}

In [8]:
splits["X_train"].describe().T[["mean", "std"]].head(10)

,mean,std
Crop_Yield,3.829500e-16,1.000013
Storage_Temperature,1.047352e-16,1.000013
Storage_Humidity,-9.921281e-17,1.000013
Fuel_Consumption,3.503869e-17,1.000013
Route_Distance,9.826067e-17,1.000013
Delivery_Time,-7.617106e-18,1.000013
Traffic_Level,-9.178613e-17,1.000013
Temperature,-3.960895e-17,1.000013
Humidity,-5.636659e-17,1.000013
Vehicle_Load_Capacity,1.188269e-16,1.000013


## Persist processed splits

Writes `X_train/X_val/X_test/y_train/y_val/y_test.csv` to `data/processed/`.

In [9]:
save_processed(splits)

## Next steps

Step 3: train baseline + tuned models per target in `src/train.py` / `notebooks/03_model_experiments.ipynb`.